# 04. Random Search

Tối ưu hai model tốt nhất ở vòng sàng lọc.

| Model | Mô tả | Số cấu hình |
|---|---|---|
| UPLIFT_RF_CIT | CausalML `UpliftRandomForestClassifier(evaluationFunction="CIT")` | 30, seed 42 |
| INT_RF | Random Forest trên `[X, W, X*W]` | 40, seed 2026 |

- Mỗi target có fold riêng: `StratifiedKFold(5, shuffle=True, random_state=42)` theo `2*W_event + y`.
- Target cash: event là Cash, control là Card. Target card thì ngược lại.
- Chọn cấu hình theo `qini_cash + qini_card` trên OOF 5-fold.

Thời gian chạy trên Kaggle CPU 4 core: khoảng 650 phút cho UPLIFT_RF_CIT và 576 phút cho INT_RF.

In [ ]:
import sys
sys.path.append("..")

from src import config as C
from src.preprocessing import load_model_data
from src.search import fold_summary, random_search, sample_configs

X, W, y_cash, y_card, feat_cols = load_model_data()
X.shape

## 1. UPLIFT_RF_CIT

In [ ]:
cit_configs = sample_configs(C.UPLIFT_RF_CIT_SPACE, C.UPLIFT_RF_CIT_BASELINE, C.UPLIFT_RF_CIT_N_ITER, C.UPLIFT_RF_CIT_SEARCH_SEED)
cit_results, cit_folds, cit_oof = random_search("UPLIFT_RF_CIT", X, W, y_cash, y_card, cit_configs, "UPLIFT_RF_CIT_CFG")
cit_results.head(10)

In [ ]:
fold_summary(cit_folds).head(10)

## 2. INT_RF

In [ ]:
int_configs = sample_configs(C.INT_RF_SPACE, C.INT_RF_BASELINE, C.INT_RF_N_ITER, C.INT_RF_SEARCH_SEED)
int_results, int_folds, int_oof = random_search("INT_RF", X, W, y_cash, y_card, int_configs, "INT_RF_RS")
int_results.head(10)

In [ ]:
fold_summary(int_folds).head(10)

## 3. Lưu kết quả

In [ ]:
cit_results.to_csv(C.RESULTS_DIR / "random_search_uplift_rf_cit.csv", index=False)
cit_folds.to_csv(C.RESULTS_DIR / "random_search_uplift_rf_cit_folds.csv", index=False)
int_results.to_csv(C.RESULTS_DIR / "random_search_int_rf.csv", index=False)
int_folds.to_csv(C.RESULTS_DIR / "random_search_int_rf_folds.csv", index=False)